# Reading and Converting Vector Files

This tutorial reads a layer of a GeoPackage with DuckDB and its spatial extension, checks what is in the file, measures its polygons in metres, and saves it as GeoParquet. It works the same for the other formats of the Geospatial Vector Files panel: only the reading line changes.

The spatial extension is installed with the Smart Regions Toolbox, so the notebook only loads it.

In [ ]:
using DuckDB, Tables

con = DBInterface.connect(DuckDB.DB)  # an in-memory database
DBInterface.execute(con, "LOAD spatial")
DBInterface.execute(con, "SET geometry_always_xy = true")  # longitude, latitude in every transform

## 1. A GeoPackage to Read

To keep the tutorial self-contained, we first write a small GeoPackage in a temporary folder: the areas within 20 km of six European cities. The buffers are drawn in metres, on ETRS89 / LAEA Europe (`EPSG:3035`), and stored in longitude and latitude (`EPSG:4326`). Replace `path` with your own `.gpkg` file to read it instead.

In [ ]:
dir = mktempdir()
path = joinpath(dir, "areas.gpkg")
cities = "SELECT name, ST_Point(lon, lat) AS geom FROM (VALUES ('Rome', 12.4964, 41.9028), ('Paris', 2.3522, 48.8566), ('London', -0.1276, 51.5072), ('Amsterdam', 4.9041, 52.3676), ('Munich', 11.582, 48.1351), ('Prague', 14.4378, 50.0755)) t(name, lon, lat)"
areas = "SELECT name, ST_Transform(ST_Buffer(ST_Transform(geom, 'EPSG:4326', 'EPSG:3035'), 20000), 'EPSG:3035', 'EPSG:4326') AS geom FROM ($cities)"
DBInterface.execute(con, "COPY ($areas) TO '$(path)' (FORMAT GDAL, DRIVER 'GPKG', LAYER_NAME 'areas', SRS 'EPSG:4326')")

## 2. What Is in the File?

`ST_Read_Meta` returns what GDAL finds in the file without reading the features: the driver, and for each layer its feature count, geometry type, coordinate system and fields. A GeoPackage can hold many layers, each in its own coordinate system.

In [ ]:
meta = Tables.columntable(DBInterface.execute(con, "SELECT driver_short_name, layers FROM ST_Read_Meta('$(path)')"))
println("Driver: ", meta.driver_short_name[1])
for layer in meta.layers[1], field in layer.geometry_fields
	println(layer.name, ": ", layer.feature_count, " features, ", field.type, ", ", field.crs.auth_name, ":", field.crs.auth_code)
end

## 3. Read the Layer

`ST_Read` reads the layer into a DuckDB table, with the geometry in a column `geom`. A `WHERE` clause keeps only some features: here those north of 49 degrees, using the minimum latitude of each polygon.

In [ ]:
DBInterface.execute(con, "CREATE OR REPLACE TABLE features AS SELECT * FROM ST_Read('$(path)', layer = 'areas') WHERE ST_YMin(geom) > 49")
tbl = Tables.columntable(DBInterface.execute(con, "SELECT name, ST_GeometryType(geom) AS type, ST_NPoints(geom)::INTEGER AS points FROM features"))

## 4. Check the Geometries

Invalid geometries, such as polygons whose boundaries cross themselves, can give wrong areas without any error. `ST_IsValid` tests each one, and `ST_MakeValid` would repair it.

In [ ]:
Tables.columntable(DBInterface.execute(con, "SELECT count(*) AS features, count(*) FILTER (WHERE NOT ST_IsValid(geom)) AS invalid FROM features"))

## 5. Measure in Metres

On longitude and latitude, `ST_Area` gives square degrees, which have no fixed size on the ground. Transforming to an equal-area system first gives square metres. Each buffer has a radius of 20 km, so its area should be close to that of a circle of that radius, about 1,257 square kilometres; the small shortfall is the polygon's corners.

In [ ]:
Tables.columntable(DBInterface.execute(con, """
SELECT name, round(ST_Area(ST_Transform(geom, 'EPSG:4326', 'EPSG:3035')) / 1e6, 1) AS area_km2,
	round(ST_Area(geom), 4) AS area_square_degrees
FROM features ORDER BY name
"""))

## 6. Save as GeoParquet

`COPY ... (FORMAT parquet)` writes the table as GeoParquet: the geometry column as binary and the `geo` metadata, which records its encoding and coordinate system. Reading it back with `read_parquet` gives a `GEOMETRY` column again.

In [ ]:
out = joinpath(dir, "areas.parquet")
DBInterface.execute(con, "COPY features TO '$(out)' (FORMAT parquet)")
geo = Tables.columntable(DBInterface.execute(con, "SELECT decode(value) AS geo FROM parquet_kv_metadata('$(out)') WHERE decode(key) = 'geo'"))
println(geo.geo[1])
Tables.columntable(DBInterface.execute(con, "SELECT name, typeof(geom) AS type FROM read_parquet('$(out)')"))

## Next

The Geospatial Vector Files panel, in Smart Regions > Places and Features, runs these steps on any GeoJSON, GeoPackage, Shapefile, GeoParquet, FlatGeobuf, KML or GPX file, and its Next Steps plot the features, compare the formats and save them as GeoJSON.